In [0]:
%python

from pyspark.sql import functions as F
df = spark.table("workspace.default.atendimentos_isa_saude")
# Considerar apenas registros com custos conhecidos
df_validos = df.filter(
    F.col("Custo (R$)").isNotNull()
)

# Indicadores financeiros por modalidade
analise_financeira = (
    df_validos
    .groupBy("Modalidade")
    .agg(
        F.count("*").alias("Registros com Custo"),
        F.round(
            F.sum("Receita Líquida (R$)"), 2
        ).alias("Receita Líquida"),
        F.round(
            F.sum("Custo (R$)"), 2
        ).alias("Custo Total"),
        F.round(
            F.avg("Custo (R$)"), 2
        ).alias("Custo Médio")
    )
    .withColumn(
        "Margem Financeira",
        F.round(
            F.col("Receita Líquida") -
            F.col("Custo Total"), 2
        )
    )
    .withColumn(
        "Margem (%)",
        F.round(
            F.when(
                F.col("Receita Líquida") != 0,
                F.col("Margem Financeira") /
                F.col("Receita Líquida") * 100
            ), 2
        )
    )
    .orderBy(F.desc("Receita Líquida"))
)

display(analise_financeira)



Modalidade,Registros com Custo,Receita Líquida,Custo Total,Custo Médio,Margem Financeira,Margem (%)
Internação Domiciliar (ID),323,1.984501642E7,1.638676871E7,50733.03,3458247.71,17.43
Assistência Domiciliar (AD),344,6286259.11,4150809.09,12066.31,2135450.02,33.97
Atendimento por Visitas,198,1008192.57,619726.66,3129.93,388465.91,38.53


In [0]:
%python

from pyspark.sql import functions as F

total = df.count()

qualidade = df.agg(
    F.count(F.when(F.col("Custo (R$)").isNull(), 1))
     .alias("Custos ausentes"),
    F.countDistinct("ID Paciente")
     .alias("Pacientes distintos"),
    F.count(F.when(F.col("Receita Líquida (R$)").isNull(), 1))
     .alias("Receitas ausentes")
)

display(qualidade)

print("Total de registros:", total)
print("Duplicidades de ID:", total - df.select("ID Paciente").distinct().count())


Custos ausentes,Pacientes distintos,Receitas ausentes
45,910,0


Total de registros: 910
Duplicidades de ID: 0


In [0]:

%sql
SELECT
    `Modalidade`,
    COUNT(*) AS total_registros,
    COUNT(`Custo (R$)`) AS registros_com_custo,
    ROUND(SUM(`Receita Líquida (R$)`), 2) AS receita_total,
    ROUND(SUM(`Custo (R$)`), 2) AS custo_conhecido,
    ROUND(
        SUM(CASE
            WHEN `Custo (R$)` IS NOT NULL
            THEN `Receita Líquida (R$)` - `Custo (R$)`
        END), 2
    ) AS margem_custos_conhecidos
FROM workspace.default.atendimentos_isa_saude
GROUP BY `Modalidade`
ORDER BY receita_total DESC;


Modalidade,total_registros,registros_com_custo,receita_total,custo_conhecido,margem_custos_conhecidos
Internação Domiciliar (ID),337,323,2.078246861E7,1.638676871E7,3458247.71
Assistência Domiciliar (AD),365,344,6676791.59,4150809.09,2135450.02
Atendimento por Visitas,208,198,1065116.06,619726.66,388465.91


In [0]:
%python

# ISA Saúde — Persistência dos indicadores em Delta Lake

# Recuperar os resultados da análise financeira em PySpark
# Renomear colunas: nomes com espaços/parênteses são inválidos em Delta
df_indicadores = analise_financeira
for antigo, novo in {
    "Registros com Custo": "registros_com_custo",
    "Receita Líquida": "receita_liquida",
    "Custo Total": "custo_total",
    "Custo Médio": "custo_medio",
    "Margem Financeira": "margem_financeira",
    "Margem (%)": "margem_pct",
}.items():
    df_indicadores = df_indicadores.withColumnRenamed(antigo, novo)

# Gravar tabela Delta no Unity Catalog
(
    df_indicadores.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.default.indicadores_financeiros_isa"
    )
)

print("Tabela Delta criada com sucesso!")

# Validar a tabela gravada
display(
    spark.table(
        "workspace.default.indicadores_financeiros_isa"
    )
)


Tabela Delta criada com sucesso!


Modalidade,registros_com_custo,receita_liquida,custo_total,custo_medio,margem_financeira,margem_pct
Internação Domiciliar (ID),323,1.984501642E7,1.638676871E7,50733.03,3458247.71,17.43
Assistência Domiciliar (AD),344,6286259.11,4150809.09,12066.31,2135450.02,33.97
Atendimento por Visitas,198,1008192.57,619726.66,3129.93,388465.91,38.53


In [0]:
%python

from pyspark.sql import functions as F

base = spark.table(
    "workspace.default.indicadores_financeiros_isa"
)

cenarios = [
    ("Base", 1.00, 1.00),
    ("Otimista", 1.05, 0.95),
    ("Pressao", 0.95, 1.10)
]

resultados = []

for nome, fator_receita, fator_custo in cenarios:
    resultado = (
        base
        .withColumn("cenario", F.lit(nome))
        .withColumn(
            "receita_simulada",
            F.round(F.col("receita_liquida") * fator_receita, 2)
        )
        .withColumn(
            "custo_simulado",
            F.round(F.col("custo_total") * fator_custo, 2)
        )
        .withColumn(
            "margem_simulada",
            F.round(
                F.col("receita_simulada") -
                F.col("custo_simulado"), 2
            )
        )
        .withColumn(
            "margem_pct_simulada",
            F.round(
                F.col("margem_simulada") /
                F.col("receita_simulada") * 100, 2
            )
        )
    )
    resultados.append(resultado)

simulacao = resultados[0]

for resultado in resultados[1:]:
    simulacao = simulacao.unionByName(resultado)

display(
    simulacao.select(
        "Modalidade",
        "cenario",
        "receita_simulada",
        "custo_simulado",
        "margem_simulada",
        "margem_pct_simulada"
    )
)


Modalidade,cenario,receita_simulada,custo_simulado,margem_simulada,margem_pct_simulada
Internação Domiciliar (ID),Base,1.984501642E7,1.638676871E7,3458247.71,17.43
Assistência Domiciliar (AD),Base,6286259.11,4150809.09,2135450.02,33.97
Atendimento por Visitas,Base,1008192.57,619726.66,388465.91,38.53
Internação Domiciliar (ID),Otimista,2.083726724E7,1.556743027E7,5269836.97,25.29
Assistência Domiciliar (AD),Otimista,6600572.07,3943268.64,2657303.43,40.26
Atendimento por Visitas,Otimista,1058602.2,588740.33,469861.87,44.39
Internação Domiciliar (ID),Pressao,1.88527656E7,1.802544558E7,827320.02,4.39
Assistência Domiciliar (AD),Pressao,5971946.15,4565890.0,1406056.15,23.54
Atendimento por Visitas,Pressao,957782.94,681699.33,276083.61,28.83


Databricks visualization. Run in Databricks to view.

In [0]:
%python

from pyspark.sql import functions as F

# Identificar a margem do cenário Base
base_margem = (
    simulacao
    .filter(F.col("cenario") == "Base")
    .select(
        "Modalidade",
        F.col("margem_simulada").alias("margem_base")
    )
)

# Comparar cada cenário com a margem Base
impacto = (
    simulacao
    .join(base_margem, on="Modalidade", how="left")
    .withColumn(
        "variacao_percentual",
        F.when(
            F.col("margem_base") != 0,
            F.round(
                (
                    F.col("margem_simulada") /
                    F.col("margem_base") - 1
                ) * 100,
                2
            )
        )
    )
    .select(
        "Modalidade",
        "cenario",
        "margem_simulada",
        "margem_base",
        "variacao_percentual"
    )
)

display(impacto)


Modalidade,cenario,margem_simulada,margem_base,variacao_percentual
Internação Domiciliar (ID),Base,3458247.71,3458247.71,0.0
Assistência Domiciliar (AD),Base,2135450.02,2135450.02,0.0
Atendimento por Visitas,Base,388465.91,388465.91,0.0
Internação Domiciliar (ID),Otimista,5269836.97,3458247.71,52.38
Assistência Domiciliar (AD),Otimista,2657303.43,2135450.02,24.44
Atendimento por Visitas,Otimista,469861.87,388465.91,20.95
Internação Domiciliar (ID),Pressao,827320.02,3458247.71,-76.08
Assistência Domiciliar (AD),Pressao,1406056.15,2135450.02,-34.16
Atendimento por Visitas,Pressao,276083.61,388465.91,-28.93


Databricks visualization. Run in Databricks to view.

In [0]:
%python

# ISA Saúde — Validação inicial com PySpark

df = spark.table("workspace.default.atendimentos_isa_saude")

print("Total de registros:", df.count())
print("Total de colunas:", len(df.columns))

display(df.limit(10))


Total de registros: 910
Total de colunas: 27


ID Paciente,Operadora,Modalidade,Patologia Principal,Região,CFS,Data Implantação,Data Alta,Status,Dias Atendimento,Diária (R$),OPME (R$),Receita Bruta (R$),Glosa (R$),Custo (R$),Reinternação,SLA Aceite (dias),Operadora Original,Chave,Operadora Padronizada,Contrato Encontrado,Custo Ausente,SLA Ausente,Data Alta Ausente,Receita Líquida (R$),Margem (R$),Margem (%)
PAC-00696,Nova Vida Seguros,Assistência Domiciliar (AD),Demência/Alzheimer,ABC,6,2026-06-28,null,Internação,86,261,0,22446,946.27,15185.45,Não,4.0,Nova Vida Seguros,NOVA VIDA SEGUROS,Nova Vida Seguros,true,false,false,true,21499.73,6314.279999999999,29.369113007465668
PAC-00557,Saúde Vida,Atendimento por Visitas,Sequela neurológica,Capital,3,2025-07-30,2025-09-06,Alta,38,151,0,5738,677.6,2897.27,Não,2.0,Saúde Vida,SAUDE VIDA,Saúde Vida,false,false,false,false,5060.4,2163.1299999999997,42.74622559481463
PAC-00473,Saúde Vida,Atendimento por Visitas,Sequela neurológica,Capital,5,2025-08-19,2025-09-26,Alta,38,168,0,6384,293.91,3742.33,Não,2.0,Saúde Vida,SAUDE VIDA,Saúde Vida,false,false,false,false,6090.09,2347.76,38.55049761169376
PAC-00860,Hospitalar Prime,Internação Domiciliar (ID),Fratura/Ortopédico,Grande SP,9,2026-05-18,2026-09-29,Alta,134,454,0,60836,5443.24,51954.9,Não,3.0,Hospitalar Prime,HOSPITALAR PRIME,Hospitalar Prime,true,false,false,false,55392.76,3437.8600000000006,6.206334546247561
PAC-00262,VITALIS SAUDE,Internação Domiciliar (ID),Sequela neurológica,Interior,6,2026-06-09,2026-09-26,Alta,109,754,0,82186,6117.98,60256.63,Não,null,VITALIS SAUDE,VITALIS SAUDE,Vitalis Saúde,true,false,true,false,76068.02,15811.390000000007,20.785857184135995
PAC-00054,Bem-Estar Seguros,Internação Domiciliar (ID),Pós-operatório,Capital,6,2026-02-01,2026-03-10,Alta,37,859,0,31783,9898.97,25509.91,Não,3.0,Bem-Estar Seguros,BEM-ESTAR SEGUROS,Bem-Estar Seguros,true,false,false,false,21884.03,-3625.880000000001,-16.568611905576812
PAC-00456,Saúde Vida,Assistência Domiciliar (AD),Pé diabético,Capital,5,2025-07-05,2025-08-15,Alta,41,351,0,14391,817.42,8442.06,Não,3.0,Saúde Vida,SAUDE VIDA,Saúde Vida,false,false,false,false,13573.58,5131.52,37.80520687983569
PAC-00761,Rede Cuidado,Internação Domiciliar (ID),DPOC,ABC,8,2026-02-22,2026-05-19,Alta,86,775,0,66650,2517.62,51427.92,Não,3.0,Rede Cuidado,REDE CUIDADO,Rede Cuidado,false,false,false,false,64132.38,12704.46,19.8097435336097
PAC-00770,Rede Cuidado,Atendimento por Visitas,Pós-operatório,Interior,6,2025-12-19,null,Internação,77,122,0,9394,444.5,null,Sim,12.0,Rede Cuidado,REDE CUIDADO,Rede Cuidado,false,true,false,true,8949.5,null,null
PAC-00737,Nova Vida Seguros,Assistência Domiciliar (AD),Insuficiência Cardíaca,Capital,6,2026-04-26,2026-05-11,Alta,15,270,0,4050,389.58,2479.56,Não,1.0,Nova Vida Seguros,NOVA VIDA SEGUROS,Nova Vida Seguros,true,false,false,false,3660.42,1180.8600000000001,32.260232432343834
